# Probe 4 (TPEx): index and corporate-event sources

Burned days only (2024-01-02, 2024-01-03). Run the single cell once and return `schema_probe4_results.zip`. This is a schema probe, not a test; it requests nothing in the P1R sample periods.

In [ ]:
# Probe 4 (TPEx): market index and corporate-event sources. Burned days 2024-01-02 and 2024-01-03 only.
# Records status, returned date, table titles and field names; raw bodies are saved for these burned days only.
# The candidate URLs other than afterTrading/otc are guesses at TPEx endpoints; a failure is an answer, not an error.
import io, json, re, time, zipfile, datetime, hashlib, requests
OUT, LOG = {}, []
H = {"User-Agent": "Mozilla/5.0 (research schema probe)", "Accept": "application/json"}
DAYS = {"20240102": ("2024/01/02", "113/01/02"), "20240103": ("2024/01/03", "113/01/03")}
def norm(s):
    s = re.sub(r"[^0-9]", "", str(s))
    if len(s) == 7: s = str(int(s[:3]) + 1911) + s[3:]
    return s
def describe(j):
    if not isinstance(j, dict): return {}
    out = {"keys": sorted(j.keys()), "date": j.get("date") or j.get("reportDate"), "tables": []}
    for t in (j.get("tables") or []):
        out["tables"].append({"title": t.get("title"), "fields": t.get("fields"), "rows": len(t.get("data") or []),
                              "head": (t.get("data") or [])[:2]})
    for k in j:
        if k.startswith("fields") or k in ("aaData",):
            out.setdefault("flat", {})[k] = j[k][:3] if isinstance(j[k], list) else j[k]
    return out
def req(name, url, params):
    t = datetime.datetime.utcnow().isoformat() + "Z"
    try:
        r = requests.get(url, headers=H, params=params, timeout=60)
        rec = {"name": name, "url": r.url, "status": r.status_code, "bytes": len(r.content),
               "sha256": hashlib.sha256(r.content).hexdigest(), "utc": t}
        try:
            j = r.json(); rec["describe"] = describe(j)
        except Exception:
            rec["describe"] = {"not_json": r.text[:300]}
        OUT[name + ".raw"] = r.content
        LOG.append(rec)
    except Exception as e:
        LOG.append({"name": name, "url": url, "params": params, "error": repr(e), "utc": t})
    time.sleep(3)

for d, (greg, roc) in DAYS.items():
    req(f"otc_{d}", "https://www.tpex.org.tw/www/zh-tw/afterTrading/otc", {"date": greg, "type": "EW", "response": "json"})
    req(f"index_inx_{d}", "https://www.tpex.org.tw/www/zh-tw/indexInfo/inx", {"date": greg, "response": "json"})
    req(f"highlight_{d}", "https://www.tpex.org.tw/www/zh-tw/afterTrading/highlight", {"date": greg, "response": "json"})
    req(f"index_legacy_{d}", "https://www.tpex.org.tw/web/stock/iNdex_info/inxh/Inx_result.php", {"l": "zh-tw", "o": "json", "d": roc})
req("exright_new", "https://www.tpex.org.tw/www/zh-tw/bulletin/exDailyQ",
    {"startDate": "2024/01/02", "endDate": "2024/01/03", "response": "json"})
req("exright_legacy", "https://www.tpex.org.tw/web/stock/exright/dailyquo/exDailyQ_result.php",
    {"l": "zh-tw", "o": "json", "d": "113/01/02", "ed": "113/01/03"})

OUT["log.json"] = json.dumps(LOG, indent=1, ensure_ascii=False).encode()
with zipfile.ZipFile("schema_probe4_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k, v in OUT.items(): z.writestr(k, v)
for e in LOG:
    dsc = e.get("describe", {})
    print(e["name"], e.get("status"), "date:", dsc.get("date"), "| tables:",
          [(t["title"], t["rows"]) for t in dsc.get("tables", [])][:6], e.get("error", ""), str(dsc.get("not_json", ""))[:80])
try:
    from google.colab import files; files.download("schema_probe4_results.zip")
except Exception: print("Saved schema_probe4_results.zip")
